# 13.1 — SOFR: contexto institucional y la reforma LIBOR→SOFR

Arranca M13. Este notebook es narrativo — la mecánica de composición y el código de
`qflib.sofr` viven en 13.2. Aquí se cubre: cómo se construye el índice SOFR, la
cronología de la reforma LIBOR→SOFR, el *discounting big bang* de octubre 2020, los
fallbacks ISDA, un enunciado conceptual de las cuatro convenciones de devengo RFR, la
metodología de CME Term SOFR, y la estacionalidad/spikes de la serie.

**Prerequisitos:** ninguno del resto del currículum (módulo autocontenido) — sí asume
comodidad con cálculo estocástico y medidas de martingala del resto del currículum.

## 1. Construcción del índice SOFR

**SOFR** (Secured Overnight Financing Rate) es una tasa de referencia *overnight*,
colateralizada por Treasuries, publicada diariamente por el Federal Reserve Bank of
New York desde el 3 de abril de 2018. A diferencia de LIBOR (una encuesta de tasas
*sin colateral* a las que un panel de bancos *dice* que podría fondearse), SOFR se
calcula a partir de transacciones *reales* de tres segmentos del mercado de repo de
Treasuries:

- **Tri-party repo** (excluyendo transacciones GCF) — el segmento más grande.
- **GCF Repo®** (General Collateral Finance Repo), compensado en DTCC.
- **Repo bilateral** liquidado en FICC's DVP service (con un filtro que excluye
  transacciones "specials" — colateral escaso que se negocia a tasas anormalmente
  bajas, no representativas del costo de fondeo general).

El volumen diario subyacente es del orden de **$1 a $1.5 trillones** de transacciones
— órdenes de magnitud más grande que las pocas transacciones (a veces ninguna) sobre
las que un banco basaba su cotización de LIBOR en los años de la manipulación. SOFR se
publica como la **mediana ponderada por volumen** de las tasas de esas transacciones,
con recortes (*trimming*) en las colas para reducir el efecto de outliers.

## 2. Cronología de la reforma

| Fecha | Evento |
|---|---|
| 2017-07-27 | La FCA (regulador de LIBOR) anuncia que no compelerá a los bancos del panel a seguir cotizando LIBOR después de 2021 |
| ~2017-2018 | El ARRC (Alternative Reference Rates Committee) se forma y elige SOFR como la tasa de reemplazo recomendada para USD |
| 2018-04-03 | Primera publicación oficial de SOFR por el NY Fed |
| 2020-10-16 | *Discounting big bang*: LCH y CME cambian el descuento de swaps de Fed Funds a SOFR el mismo fin de semana |
| 2021-03-05 | ISDA fija (congela) el spread de fallback LIBOR→SOFR |
| 2021-12-31 | Cesación de LIBOR USD en los tenors *no representativos* (1 semana, 2 meses) |
| 2023-06-30 | Cesación del resto de los tenors de LIBOR USD (overnight, 1M, 3M, 6M, 12M) |

La ventana entre el anuncio (2017) y la cesación completa (2023) — casi seis años —
refleja la escala del problema: LIBOR era la tasa de referencia de un estimado de
$200+ trillones en contratos vigentes (swaps, préstamos, bonos flotantes, hipotecas)
en el momento del anuncio.

## 3. El *discounting big bang* (octubre 2020)

El 16 de octubre de 2020, LCH y CME cambiaron simultáneamente la curva de descuento
usada para valuar **todos** los swaps de tasa de interés en USD compensados
centralmente: de Fed Funds Effective Rate (OIS) a SOFR. Esto no fue un cambio
cosmético — cambiar el numerario (la curva de descuento) cambia el valor presente de
*todo* flujo futuro, incluso si la tasa flotante subyacente del swap no cambió.

**Mecánica de la compensación en efectivo (cash compensation):** para neutralizar el
cambio de valor en cada posición vigente en el momento del switch, las cámaras de
compensación calcularon la diferencia de valuación (descontado a Fed Funds vs
descontado a SOFR) para cada swap y pagaron/cobraron esa diferencia en efectivo a cada
contraparte, dejando el valor económico neto sin cambio en el momento del switch —
sólo el *numerario* futuro cambió.

**Basis swaps compensatorios:** dado que la curva de descuento SOFR y la curva de
proyección de la tasa flotante (que para swaps SOFR-SOFR coinciden, pero para swaps
legacy referenciados a Fed Funds o LIBOR no) se movieron de forma distinta, el mercado
usó basis swaps Fed Funds-SOFR para cubrir la exposición residual generada por el
cambio de curva.

**Impacto en swaptions vigentes:** una swaption ATM (at-the-money) tiene su strike
fijado en el momento de la emisión, referenciado al forward swap rate calculado con la
curva de descuento *de ese momento*. Cuando el numerario cambia, el forward swap rate
implícito cambia (aunque las tasas de mercado no se hayan movido), así que el "ATM" de
ayer deja de ser exactamente ATM hoy — el valor de mercado de la swaption se mueve por
el solo hecho del cambio de numerario, no por movimiento del mercado subyacente.

## 4. Fallbacks ISDA

Para los contratos de derivados OTC que referenciaban LIBOR y no tenían lenguaje de
fallback robusto (o donde el lenguaje existente asumía que LIBOR seguiría existiendo),
ISDA publicó un protocolo de fallback estándar:

- **Spread de fallback**: para 3M USD LIBOR, **26.161 puntos base**. Este spread no es
  arbitrario — es la **mediana histórica** del *basis* entre LIBOR y el SOFR compuesto
  *in-arrears* a 5 años, calculada sobre un periodo de lookback fijo y **congelada** el
  día del anuncio de cesación (2021-03-05), precisamente para evitar que el spread
  pudiera manipularse o ser objeto de arbitraje una vez que la fecha de cesación fuera
  de conocimiento público.
- El fallback usa SOFR compuesto **in-arrears** (no Term SOFR forward-looking) más
  este spread, como la tasa de referencia de reemplazo.
- Para las opciones sobre eurodólar (referenciadas a 3M LIBOR), el ajuste de conversión
  a SOFR incorpora un ajuste de **strike de 25 puntos base**, parte de la migración de
  ese mercado a futuros de SOFR.

## 5. Las cuatro convenciones de devengo (enunciado conceptual)

SOFR es una tasa *overnight*, así que un periodo de devengo de, por ejemplo, 3 meses
requiere **componer** ~63 fixings diarios. Cómo se alinean esos fixings con el periodo
de pago real da lugar a cuatro convenciones de mercado — la mecánica exacta (fórmulas,
código, comparación contra datos reales del NY Fed) está en **13.2**:

- **Lookback**: la *tasa* usada cada día se toma de $X$ días hábiles antes, pero el
  *peso* (cuántos días calendario cubre) sigue siendo el del calendario original del
  periodo.
- **Lockout**: los últimos $X$ días hábiles del periodo *congelan* la tasa en el valor
  observado $X$ días hábiles antes del fin — evita que el pagador no pueda calcular su
  flujo con suficiente antelación para procesarlo.
- **Observation shift**: tanto la tasa *como* el peso vienen de una ventana completa
  desplazada $X$ días hábiles hacia atrás — a diferencia de lookback puro, esto sí
  corresponde exactamente a "la curva evaluada en la ventana desplazada".
- **Rate cutoff**: mecánicamente idéntico a lockout (congela los últimos días), nombre
  distinto por convención de mercado — típicamente asociado a bonos de tasa flotante
  más que a swaps.

Cada convención cambia el *payoff* del instrumento: no son equivalentes entre sí ni
frente al in-arrears puro, y la elección de convención es un término negociado del
contrato, no un detalle operativo neutral.

## 6. CME Term SOFR

A diferencia del SOFR compuesto *backward-looking* (que sólo se conoce con certeza al
final del periodo), **CME Term SOFR** es una tasa *forward-looking*, publicada en
tenors 1M/3M/6M/12M, pensada para instrumentos (como préstamos sindicados) donde los
participantes necesitan conocer la tasa *al inicio* del periodo.

**Metodología**: Term SOFR se deriva de los precios de **futuros de SOFR** listados en
CME — específicamente, 13 futuros SOFR de 1 mes consecutivos y 5 futuros SOFR de 3
meses consecutivos. La curva de futuros implica expectativas de mercado sobre la
trayectoria de SOFR compuesto, de la cual se extrae la tasa Term SOFR para cada tenor
mediante un algoritmo de suavizado (waterfall) documentado en el CME Term SOFR
Reference Rates Benchmark Methodology.

El uso de Term SOFR está explícitamente **restringido por ARRC** principalmente a
préstamos de negocio (business loans) y algunos derivados de cobertura de esos
préstamos — no está recomendado como sustituto general del SOFR compuesto in-arrears
en el grueso del mercado de swaps y derivados, precisamente porque re-introduce el
mismo problema de fondo que llevó al abandono de LIBOR: una tasa forward-looking que
requiere estimar/proyectar en vez de observar transacciones reales ya ocurridas.

## 7. Estacionalidad y spikes

SOFR, al estar atado a transacciones reales de repo colateralizado con Treasuries, es
sensible a la oferta y demanda de colateral y de reservas bancarias — algo que Fed
Funds (una tasa de fondeo *sin colateral* entre bancos) no refleja de la misma forma.
Patrones observados:

- **Fin de trimestre / fin de año**: los bancos reducen su actividad de balance por
  requerimientos regulatorios (leverage ratio, SLR), lo que puede reducir la oferta de
  financiamiento repo justo quienes más colateral necesitan financiar — empujando la
  tasa al alza en esos días puntuales.
- **La escasez de reservas de septiembre de 2019**: el evento de referencia. SOFR
  saltó de ~2.2% a picos intradía reportados sobre 5-10% (y el propio SOFR publicado
  llegó a 5.25% el 17 de septiembre de 2019) cuando una combinación de pagos de
  impuestos corporativos y liquidación de subastas de Treasuries drenó reservas
  bancarias simultáneamente, mientras la oferta de repo no se ajustó con la rapidez
  suficiente. La Fed respondió reanudando operaciones de repo permanentes. Este
  episodio es la razón por la que, en retrospectiva, SOFR es estructuralmente más
  volátil día a día que Fed Funds: refleja directamente la mecánica de financiamiento
  del colateral, no un promedio suavizado de cotizaciones bancarias.

## Validación

Dos verificaciones de consistencia interna sobre las cifras citadas arriba — no es un
cálculo de modelo, es una validación de que las citas numéricas y la cronología están
bien transcritas en dos representaciones independientes.

In [1]:
from datetime import date

# La cita del spread de fallback (26.161 bp) verificada en dos representaciones
# independientes: puntos base y decimal.
FALLBACK_SPREAD_3M_LIBOR_BP = 26.161  # ARRC/ISDA, congelado 2021-03-05

assert abs(FALLBACK_SPREAD_3M_LIBOR_BP / 100 / 100 - 0.0026161) < 1e-12

# La cronología de la reforma debe ser estrictamente ascendente en el tiempo.
timeline = [
    date(2017, 7, 27),   # anuncio FCA
    date(2018, 4, 3),    # primera publicación SOFR por el NY Fed
    date(2020, 10, 16),  # discounting big bang (CME/LCH)
    date(2021, 3, 5),    # ISDA congela el fallback spread
    date(2021, 12, 31),  # cesación LIBOR USD tenors no representativos
    date(2023, 6, 30),   # cesación LIBOR USD resto de tenors
]
assert timeline == sorted(timeline)
print("OK: spread y cronología consistentes.")

OK: spread y cronología consistentes.


## Referencias

- ARRC — *SOFR: A Year in Review* y guías de convención de devengo RFR.
- CME Group — *Term SOFR Reference Rates Benchmark Methodology*; CME Rulebook
  capítulos 460 y 900/902.
- Klingler, S. & Syrstad, O. (2021). *Life After LIBOR*. Journal of Financial
  Economics, 141(2).
- Henrard, M. (2019). *LIBOR Fallback and Quantitative Finance*. Risks, 7(3), 88.
- Henrard, M. *Discounting Transition: Big Bang Impacts*. SSRN 3530464.

Siguiente: **13.2 — convenciones de devengo**, con la mecánica exacta de composición
(`qflib.sofr.compound_sofr`) y el entregable que compara el resultado contra el SOFR
Average publicado por el NY Fed.